## Setup 

In [ ]:
# Load pipeline utils
source("~/workspace/pipelines/snt_dhis2_population_transformation/utils/snt_dhis2_population_transformation.r") 
snt_paths <- init_snt_workspace(
    snt_pipeline_name="snt_dhis2_population_transformation",
    packages=c("arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "httr", "glue", "reticulate"))

# config file path
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, "SNT_config.json"))

# output path
POPULATION_DATA_PATH <- file.path(snt_paths$DATA_PATH , "dhis2", "population_transformed")

# Save config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
format_dataset_id <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

# Fix metadata column names
metadata_columns <- c("YEAR", "ADM1_NAME", "ADM1_ID", "ADM2_NAME", "ADM2_ID")
population_column <- "POPULATION"   

#### Validate parameters

In [ ]:
if(!exists("TOT_POP_REFERENCE")) TOT_POP_REFERENCE <- NULL
if(!exists("TOT_POP_REFERENCE_YEAR")) TOT_POP_REFERENCE_YEAR <- NULL
if(!exists("GROWTH_FACTOR")) GROWTH_FACTOR <- NULL
if(!exists("GROWTH_REFERENCE_YEAR")) GROWTH_REFERENCE_YEAR <- NULL
if(!exists("POP_UNDER_5")) POP_UNDER_5 <- NULL
if(!exists("POP_PREGNANT_WOMEN")) POP_PREGNANT_WOMEN <- NULL
if(!exists("POP_0_1_Y")) POP_0_1_Y <- NULL
if(!exists("POP_1_2_Y")) POP_1_2_Y <- NULL
if(!exists("POP_5_10_Y")) POP_5_10_Y <- NULL
if(!exists("POP_5_36_M")) POP_5_36_M <- NULL
if(!exists("POP_50_PLUS")) POP_50_PLUS <- NULL

if(!exists("DISAGGREGATION_FILE")) DISAGGREGATION_FILE <- NULL

## Load DHIS2 population data

-Load DHIS2 population from latest formatted dataset version.

In [ ]:
dhis2_population <- load_dataset_file(format_dataset_id, paste0(COUNTRY_CODE, "_population.parquet"), verbose=TRUE)
dhis2_population <- dhis2_population %>% select(all_of(c(metadata_columns, population_column)))  # Select only total population, ignore the other columns..
dim(dhis2_population)
head(dhis2_population, 3)

## SNT population scaling

Adjust DHIS2 population using 'TOT_POP_REFERENCE' as scaling factor (optional).  
If a value is provided, we compute and replace the population column with the adjusted values.

Details:  
    - The **scaled population** is stored in a temporary column "POPULATION_SCALED", which is later removed.  
    - The **scaled population** values replaces the values in the **POPULATION** column.

In [ ]:
# Check for available data to perform adjustment (more work!!!!)

if (is.null(TOT_POP_REFERENCE)) {
    log_msg(glue("Total population reference not available. Population adjustment skipped."), "warning")
} else {

    log_msg("Applying population adjustment..")
    
    # handle pop reference year 
    year_selected <- resolve_reference_year(unique(dhis2_population$YEAR), TOT_POP_REFERENCE_YEAR)
    
    year_selected_total <- dhis2_population %>%
        filter(YEAR == year_selected) %>%
        summarise(total_year_pop = sum(.data[[population_column]], na.rm = TRUE)) %>%
        pull(total_year_pop)
    
    scaling_factor = year_selected_total / TOT_POP_REFERENCE
    dhis2_population[[population_column]] <- as.integer(round(dhis2_population[[population_column]] * scaling_factor))
        
    log_msg(glue("Total population year {year_selected}: {scales::comma(year_selected_total)} | Reference: {scales::comma(TOT_POP_REFERENCE)} | Scaling factor: {round(scaling_factor, 4)}"))
    log_msg(glue("Population column '{population_column}' has been replaced with scaled values."))
    head(dhis2_population,3)    
}

## SNT Population disaggregations

Any defined disaggregations will be computed from the 'POPULATION_DISAGGREGATIONS' in the configuration file and included as additional columns in the final table.  

### Population under 5 disaggregation

In [ ]:
if (!is.null(POP_UNDER_5)) {
    if ("POP_UNDER_5" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_UNDER_5' already exists and will be overwritten using the provided disaggregation proportion: {POP_UNDER_5}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_UNDER_5', factor: {POP_UNDER_5}"))
    dhis2_population[["POP_UNDER_5"]] <- round(dhis2_population[["POPULATION"]] * POP_UNDER_5)
    head(dhis2_population, 3)
}

### Population pregnant women disaggregation

In [ ]:
if (!is.null(POP_PREGNANT_WOMEN)) {
    if ("POP_PREGNANT_WOMEN" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_PREGNANT_WOMEN' already exists and will be overwritten using the provided disaggregation proportion: {POP_PREGNANT_WOMEN}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_PREGNANT_WOMEN', factor: {POP_PREGNANT_WOMEN}"))
    dhis2_population[["POP_PREGNANT_WOMEN"]] <- round(dhis2_population[["POPULATION"]] * POP_PREGNANT_WOMEN)
    head(dhis2_population, 3)
}

### Population 0 to 1 years disaggregation

In [ ]:
if (!is.null(POP_0_1_Y)) {
    if ("POP_0_1_Y" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_0_1_Y' already exists and will be overwritten using the provided disaggregation proportion: {POP_0_1_Y}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_0_1_Y', factor: {POP_0_1_Y}"))
    dhis2_population[["POP_0_1_Y"]] <- round(dhis2_population[["POPULATION"]] * POP_0_1_Y)
    head(dhis2_population, 3)
}

### Population 1 to 2 years disaggregation

In [ ]:
if (!is.null(POP_1_2_Y)) {
    if ("POP_1_2_Y" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_1_2_Y' already exists and will be overwritten using the provided disaggregation proportion: {POP_1_2_Y}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_1_2_Y', factor: {POP_1_2_Y}"))
    dhis2_population[["POP_1_2_Y"]] <- round(dhis2_population[["POPULATION"]] * POP_1_2_Y)
    head(dhis2_population, 3)
}

### Population 5 to 10 years disaggregation

In [ ]:
if (!is.null(POP_5_10_Y)) {
    if ("POP_5_10_Y" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_5_10_Y' already exists and will be overwritten using the provided disaggregation proportion: {POP_5_10_Y}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_5_10_Y', factor: {POP_5_10_Y}"))
    dhis2_population[["POP_5_10_Y"]] <- round(dhis2_population[["POPULATION"]] * POP_5_10_Y)
    head(dhis2_population, 3)
}

### Population 5 to 36 months disaggregation

In [ ]:
if (!is.null(POP_5_36_M)) {
    if ("POP_5_36_M" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_5_36_M' already exists and will be overwritten using the provided disaggregation proportion: {POP_5_36_M}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_5_36_M', factor: {POP_5_36_M}"))
    dhis2_population[["POP_5_36_M"]] <- round(dhis2_population[["POPULATION"]] * POP_5_36_M)
    head(dhis2_population, 3)
}

### Population 50 plus years disaggregation

In [ ]:
if (!is.null(POP_50_PLUS)) {
    if ("POP_50_PLUS" %in% colnames(dhis2_population)) {
        log_msg(glue("Column 'POP_50_PLUS' already exists and will be overwritten using the provided disaggregation proportion: {POP_50_PLUS}."), "warning")
    }
    
    log_msg(glue("Adding disaggregation 'POP_50_PLUS', factor: {POP_50_PLUS}"))
    dhis2_population[["POP_50_PLUS"]] <- round(dhis2_population[["POPULATION"]] * POP_50_PLUS)
    head(dhis2_population, 3)
}

## SNT Population disaggregations file based

Any disaggregation proportions provided in the user-edited **COUNTRY_CODE_population_transformation_template.csv** will be used to compute additional population columns in the final output table.  
  
When processing this template, the system will automatically apply the following rules:  

**Calculations**: The new columns are generated by multiplying the total population by the provided disaggregation proportion in the file. The final results are automatically rounded to the nearest whole number.  
**Skipping Blank Columns**: You do not need to fill out disaggregation column in the template. If a specific disaggregation column is left completely blank, the system will safely ignore it and exclude it from the final table.


In [ ]:
# add disagregation
if(!is.null(DISAGGREGATION_FILE)) {
    log_msg("Applying population disaggregation based on CSV file..")
    log_msg(glue("Loading disaggegation proportions file: {DISAGGREGATION_FILE}"))
    disaggregation_data <- load_csv_file(DISAGGREGATION_FILE)
    dhis2_population <- add_population_disaggregations(dhis2_population, disaggregation_data)
}
head(dhis2_population, 3)

## SNT Population projection and back-calculation using a growth factor  
  
Projects population figures backward and forward in time using the specified annual growth rate and reference year, across all **available population indicator columns**.  
Values for overlapping years will be overwritten.


In [ ]:
if (is.null(GROWTH_FACTOR)) {
    log_msg("No 'Growth factor' provided. Population projection skipped.")
    
} else {

    log_msg("Applying population projections..")
    n_years_future <- 6 # n_years to the future 
    n_years_past <- 6 # n_years to the past 

    # Current population indicator columns
    value_columns <- setdiff(colnames(dhis2_population), metadata_columns)
    log_msg(glue("Population indicators in dataset: {paste0(value_columns, collapse=', ')}"))
        
    # Set reference_year to the max year if NULL or not present    
    reference_year <- resolve_reference_year(unique(dhis2_population$YEAR), GROWTH_REFERENCE_YEAR)
    log_msg(glue("Applying growth factor {GROWTH_FACTOR} to project {paste0(value_columns, collapse=', ')} from reference year: '{reference_year}'."))

    projection_years_backward <- seq(reference_year - 1, reference_year - n_years_past, by=-1)
    projection_years_forward <- seq(reference_year + 1, reference_year + n_years_future)
    overlapping_years <- intersect(unique(dhis2_population$YEAR), projection_years_backward)
    if (length(overlapping_years) > 0) {
        log_msg(
            glue("The following years already exist and will be overwritten by projections from {reference_year}: {paste(overlapping_years, collapse = ', ')}"), 
            "warning")
    }

    dhis2_population_reference <- dhis2_population[dhis2_population$YEAR == reference_year, ]
    
    # Project backwards
    population_backward <- project_backward(ref_data=dhis2_population_reference,
                                            years=projection_years_backward,
                                            growth_factor=GROWTH_FACTOR,
                                            target_columns=value_columns)
    log_msg(glue("Computed backward projected values for years: {paste(unique(population_backward$YEAR), collapse = ', ')}"))
    
    # Project forwards
    population_forward <- project_forward(ref_data=dhis2_population_reference, 
                                          years=projection_years_forward,
                                          growth_factor=GROWTH_FACTOR,
                                          target_columns=value_columns)
    log_msg(glue("Computed forward projected values for years: {paste(unique(population_forward$YEAR), collapse = ', ')}"))

    # Bind results and replace 'dhis2_population'
    dhis2_population <- bind_rows(dhis2_population_reference, population_backward, population_forward) %>%  arrange(YEAR)
        
}

In [ ]:
# Check total populations per year
all_years <- sort(unique(dhis2_population$YEAR))
n_units <- nrow(dhis2_population[dhis2_population$YEAR == max(all_years), ])
log_msg(glue::glue("Total yearly population of {n_units} organisation units at Administrative level 2."))

for (year in sort(unique(dhis2_population$YEAR))) {
    year_data <- dhis2_population[[population_column]][dhis2_population$YEAR == year]    
    tot_pop <- sum(year_data, na.rm = TRUE)
    log_msg(glue("Total population year {year} : {format(tot_pop, big.mark=',')}"))
}

## Output formatted population data

In [ ]:
# write parquet file
write_parquet(dhis2_population, file.path(POPULATION_DATA_PATH, paste0(COUNTRY_CODE, "_population.parquet")))
write.csv(dhis2_population, file.path(POPULATION_DATA_PATH, paste0(COUNTRY_CODE, "_population.csv")), row.names = FALSE)

# log
log_msg(glue("Transfomerd population data saved under: {file.path(POPULATION_DATA_PATH, paste0(COUNTRY_CODE, '_population.csv'))}"))

### Data Summary 

In [ ]:
# Data summary
print(summary(dhis2_population))